In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import seaborn as sns
from IPython.display import display

In [ ]:
TASK = Path.cwd().parent
INPUT, OUTPUT = TASK/'input', TASK/'output'
STEM = 'pnas_dblp_ar1_stage_parameters'
sns.set_theme(style='ticks', context='paper', palette='colorblind',
              rc={'pdf.fonttype':42, 'axes.labelsize':7.5, 'axes.titlesize':8,
                  'xtick.labelsize':6, 'ytick.labelsize':7, 'axes.linewidth':.6,
                  'xtick.major.size':2.5, 'ytick.major.size':2.5,
                  'xtick.major.width':.6, 'ytick.major.width':.6})
COLORS = sns.color_palette('colorblind')
LEFT, RIGHT = COLORS[0], COLORS[3]

In [ ]:
params = pd.read_csv(INPUT/'simulation/fit/ar1_s/parameters.csv').sort_values('start')
# stationary log-offset mean with stage coefficients held fixed and no zero floor
params['implied_log_mean'] = (params.intercept/(1-params.beta)).where(params.beta.abs().lt(1))
display(params[['stage','intercept','beta','sigma','implied_log_mean']])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(3.42,1.65), sharex=True, layout='constrained')
fig.get_layout_engine().set(w_pad=.025, h_pad=.025, wspace=.035)
panels = [
    ('A  Coefficients', 'intercept', r'$\alpha$', 'beta', r'$\beta$'),
    ('B  Shock and mean', 'sigma', r'Shock SD, $\sigma$', 'implied_log_mean', r'$\alpha/(1-\beta)$'),
]
for ax, (title,left,left_label,right,right_label) in zip(axes,panels):
    twin = ax.twinx()
    sns.lineplot(data=params, x='stage', y=left, ax=ax, color=LEFT,
                 marker='o', markersize=3.5, linewidth=1.15, estimator=None, errorbar=None)
    sns.lineplot(data=params, x='stage', y=right, ax=twin, color=RIGHT,
                 marker='s', markersize=3.5, linewidth=1.15, linestyle='--',
                 estimator=None, errorbar=None)
    ax.set(xlabel='', ylabel=left_label)
    twin.set(ylabel=right_label)
    ax.set_title(title, loc='left', fontweight='bold')
    for side, name, color in [(ax,'left',LEFT), (twin,'right',RIGHT)]:
        side.margins(y=.14)
        side.yaxis.set_major_locator(MaxNLocator(nbins=3, steps=[1,2,2.5,5,10]))
        side.yaxis.label.set_color(color)
        side.yaxis.labelpad = 2
        side.tick_params(axis='y', colors=color, pad=1.5)
        side.spines[name].set_color(color)
    sns.despine(ax=ax, right=True)
    sns.despine(ax=twin, left=True, right=False, bottom=True)
fig.supxlabel('Transition-start years', fontsize=7.5)
for extension in ('pdf','png'):
    fig.savefig(OUTPUT/f'{STEM}.{extension}', dpi=300, facecolor='white')
plt.show()
plt.close(fig)